In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt

from HHO_kernel import HHO_kernel
from test_utils import test_HHO_convergence

plt.rcParams.update({
    # Make tick labels point inside
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    # Set thicker axis lines (spines)
    'axes.linewidth': 1.5,
})

One of the particularities of the 1D case is that all choices of polynomial degree $k$ for the cell unknowns lead to the same transmission problem to couple the face unknowns. 
We can thus implement and validate the transmission problem independently of the implementation of the cell problems.

More specifically, the transmission problem is the same as the discretization of the conforming $\mathbb{P}_1$ method and we know that the solution to this problem is equal to the analytical solution $u$ at the faces (i.e., the vertices in 1D) when there is no discretization error in the right-hand side of the discrete problem (i.e., the integrals $\int_T \Delta u \cdot v$ are computed exactly on all cells $T$ and for all test functions $v$). This is the case when $u$ is a polynomial of degree at most 2.

# Validation of the transmission problem
Let us define a few test cases. The grid we use is very coarse on purpose, because this allows us to visualize discontinuities of the HHO reconstruction for the $k=0$ case more clearly below.

In [ ]:
# Define grid for test cases
xL = 0
xR = 1
N_cells = 4
xx = np.linspace(xL, xR, N_cells+1)

Each test case is defined by the exact solution `u`, its negative second derivative `neg_neg_ddu`, the type of boundary conditions to prescribe on the left and on the right `bc` (`'D'` for Dirichlet, `'N'` for Neumann), the values of the boundary conditions when applicable through `bc_left` and `bc_right` (note that Neumann conditions are always homogeneous) and the average `average` in case of `NN` boundary conditions.

In [ ]:
# Define test cases
def test_case(number):
    match number:
        case 1:
            ux = "u(x) = x"
            u = lambda x: x
            neg_ddu = lambda x: 0
            bc = 'DD'
            bc_left = 0
            bc_right = 1
            average = None
        case 2:
            ux = "u(x) = -4*(x^2 - x)"
            u = lambda x: -4*(x**2 - x) 
            neg_ddu = lambda x: 8
            bc = 'DD'
            bc_left = 0
            bc_right = 0
            average = None
        case 3:
            ux = "u(x) = 1 - x^3"
            u = lambda x: -x**3 + 1
            neg_ddu = lambda x: 6*x
            bc = 'ND'
            bc_left = None
            bc_right = 0
            average = None
        case 4:
            ux = "u(x) = 4 + cos(pi*x)"
            u = lambda x: 4 + np.cos(np.pi*x)
            neg_ddu = lambda x: np.pi**2 * np.cos(np.pi*x)
            bc = 'NN'
            bc_left = None
            bc_right = None
            average = 4
        case _:
            raise ValueError(f"No available test case to match number {number}.")
    message = f"Test case {number}:"
    print(f"{message} {ux}")
    return u, neg_ddu, bc, bc_left, bc_right, average

The function below will be used to plot the results of the transmission problem.

In [ ]:
# Define function to plot the result of the k=0 HHO approximation
def test_HHO(solver, test_number, *plot_args):
        u, neg_ddu, bc, bc_left, bc_right, average = test_case(test_number)
        bc_args = {}
        if bc_left is not None:
                bc_args["bc_left"] = bc_left 
        if bc_right is not None:
                bc_args["bc_right"] = bc_right
        if average is not None:
                bc_args["average"] = average
        # Solve Poisson equation with HHO
        solver.boundary_conditions = bc
        solver.solve(neg_ddu, **bc_args)
        # Compute max norm error of the transmission problem
        max_difference = np.max(np.abs(u(solver.points) - solver.solution_face))
        print(f"Max norm error of HHO at the faces = {max_difference:.2e}.")
        # Plot the exact solution and the HHO approximation
        fig, ax = plt.subplots(figsize=(4,3))
        xL = np.min(solver.points)
        xR = np.max(solver.points)
        xx = np.linspace(xL,xR,101)
        ax.plot(xx, 
                u(xx), 
                'k-', 
                linewidth=1, 
                label="Solution")
        solver.plot(ax, *plot_args)
        ax.tick_params(direction='in')
        ax.grid(True, 
                which='major', 
                linestyle='--', 
                linewidth=0.5, 
                color='lightgrey')
        ax.legend(loc='center left', 
                bbox_to_anchor=(1, 0.5))
        plt.show()

The HHO-solver for the Poisson equation on the grid `xx` is initialized as follows.

In [ ]:
# Build HHO solver for the Poisson equation (the default cell degree used is 0)
poisson = HHO_kernel(xx)

### Transmission problem for an affine function

In [ ]:
test_HHO(poisson, 1, "faces") # "faces" indicates that we want to plot the solution at the faces

Indeed, the solution at the faces reproduces the exact solution. This is still the case if the exact solution is a parabola, as shows the next example.

### Transmission problem for a quadratic function

In [ ]:
test_HHO(poisson, 2, "faces")

The next example shows that this is no longer the case for a 3rd degree polynomial, but it does show that Neumann conditions are implemented correctly at the left boundary.

### Transmission problems that do not reproduce the analytical solution

In [ ]:
test_HHO(poisson, 3, "faces")

We show a last example that is defined with Neumann conditions on both sides of the domain.

In [ ]:
test_HHO(poisson, 4, "faces")

# Full implementation for cell unkowns of degree 0
We also fully implement the $k=0$ case because both the value of the cell unknowns and the reconstruction of the potential of degree 1 can be directly computed in terms of the face unknowns, which are the solution of the transmission problem that we considered above.

## Cell unknowns
If the grid consists of $N$ cells numbered from $1$ to $N$ and the cell unknowns are $\lambda_0, \dots, \lambda_N$, the face unknown $u_i$ on the $i$-th cell is given by 
$$
u_i = \frac{1}{2} h_i^2 \overline{f}_i + \frac{1}{2}(\lambda_{i-1} + \lambda_i),
$$
where $h_i$ is the length of the cell and $\overline{f}_i$ is the average of the source $f = -u''$ of the Poisson equation. Of course, when $f$ is an arbitrary function, $\overline{f}_i$ will be an approximation of the average on the cell in the actual computations. 
In our implementation, we use the value of $f$ at the barycenter of the cell, since this lowest-order approximation is sufficient to attain the theoretical optimal convergence order.

The cell unknowns can be visualized on top of any one of the foregoing test cases.

In [ ]:
test_HHO(poisson, 2, "faces", "cells")

## Reconstruction
The reconstruction can directly be computed from the cell and face unknowns. 
Let $x^b_i$ denote the barycenter of the $i$th cell $(x_{i-1},x_i)$ (so $x^b_i = \frac{1}{2}(x_{i-1}+x_i)$), then the reconstruction $R$ on the $i$-th cell is given by
$$
R\vert_{(x_{i-1},x_i)}(x) = u_i + \frac{\lambda_i - \lambda_{i-1}}{h_i} (x - x^b_i).
$$
This is indeed the unique affine function on $(x_{i-1},x_i)$ satisfying $\frac{1}{h_i} \int_{x_{i-1}}^{x_i} R = u_i$ and $R'\vert_{(x_{i-1},x_i)}(x) = \frac{\lambda_i - \lambda_{i-1}}{h_i}$.

As predicted by the theory, we now verify that the reconstruction reproduces the exact solution when it is affine.

In [ ]:
test_HHO(poisson, 1) # equivalent to test_HHO(poisson, 1, "faces", "cells", "reconstruction")

When $u$ is a quadratic function, the HHO reconstruction (a first-order polynomial) cannot be equal to the exact solution. In this case, the reconstruction looks as follows.

In [ ]:
test_HHO(poisson,2)

It might be surprising at first sight that the reconstruction is in fact continuous (there is no guarantee for this to be the case when $k=0$), but that it is *not* equal to $u$ at the faces.
This is explained as follows. 
We know that the face unknowns are equal to $u$ at the faces. If the reconstruction were equal to the function $\tilde{R}$ defined by 
$$
\tilde{R}\vert_{(x_{i-1},x_i)}(x) = \frac{\lambda_{i}+\lambda_{i-1}}{2} + \frac{\lambda_i - \lambda_{i-1}}{h_i} (x - x^b_i),
$$
it would simply be equal to the (continuous) $\mathbb{P}_1$ interpolation of $u$. 
However, from the definition of $u_i$ it follows that 
$$
(R-\tilde{R})\vert_{(x_{i-1},x_i)} = \frac{1}{2} h_i^2 \overline{f}_i,
$$
and thus the reconstruction is only equal to the $\mathbb{P}_1$ interpolation on the $i$-th cell if the average of $f$ vanishes.
In particular, we recover the exact solution if $u$ is affine, which implies $f=0$.

Now, for test case 2, $f$ (and hence $f_i$) is constant in each cell. 
Moreover, we use a homogeneous grid, i.e., $h_i$ is constant as well. 
Then the above shows that $R$ differs from the $\mathbb{P}_1$ interpolation by the same amount on each cell (namely $\frac{1}{2} h_i^2 \overline{f}_i$), and remains continuous. 
This explains why the reconstruction on the figure above looks the way it does.

When we use an inhomogeneous grid, the reconstruction is no longer continuous, as the following example shows.
More precisely, we see that the offset with respect to the $\mathbb{P}_1$ interpolation becomes larger on larger cells and smaller on smaller cells, in correspondence with the factor $h_i^2$ in $\frac{1}{2} h_i^2 \overline{f}_i$.

In [ ]:
# Define an inhomogeneous grid
yy = copy.deepcopy(xx)
# Move every other point to the right by a quarter of the original cell size
yy[range(1,N_cells,2)] += 0.25*(xR-xL)/N_cells 
# Define a new solver for this grid
poisson_inhomogeneous_grid = HHO_kernel(yy)
test_HHO(poisson_inhomogeneous_grid, 2)

We end this section by showing the HHO approximation on the other examples considered above.

In [ ]:
test_HHO(poisson, 3)

In [ ]:
test_HHO(poisson, 4)

# Convergence
For any $f \in L^2(\Omega)$, let $u$ denote the solution to $-u'' = f$ with some choice of boundary conditions and let $u_H$ be the potential reconstruction of the HHO approximation of $u$ (computed from the sole knowledge of $f$).
It has been shown that
$$
\lVert u - u_H \rVert_{L^2(\Omega)} \leq C h^{2},
$$
and
$$
\lVert u - u_H \rVert_{H^1(\Omega)} \leq C h.
$$
for any sufficiently regular function $f$ and the $H^1$-norm is to be understood in the broken sense, because the reconstructed potential $u_H$ is, in general, discontinuous. We confirm these convergence rates below in the case where $u(x) = 4 + \cos(\pi x)$.

In [ ]:
test_HHO_convergence("Poisson solve")